# To Run Spectral Ratio Illumination Demo on Google Colab

Omar Elmady 

Wednesday, Dec 11 

CS 7180 

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Update model link** in Step 2 if you have it from your professor
3. **Run all cells in order** (Runtime → Run all)

## What this notebook does:
- Checks GPU availability
- Downloads model from Google Drive (if link provided)
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "No GPU detected - will use CPU (slower but works)"

## Step 2: Clone Repository and Download Model

### 2a. Clone Repository

First, clone the GitHub repository to get all the code and data.

In [ ]:
import os

# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
print("Cloning repository from GitHub...")
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

print("\nRepository cloned successfully!")
print("\nRepository contents:")
!ls -lh

### 2b. Download Model from Google Drive

To extract file ID from a Drive link like `https://drive.google.com/file/d/1ABC123XYZ/view?usp=sharing`, copy the `1ABC123XYZ` part. If you need to change it, update `MODEL_DRIVE_ID` below.  

The model will download directly into the repository's `model/` folder.

In [ ]:
import os

# ============================================
# CONFIGURATION: Google Drive file ID for model
# ============================================
MODEL_DRIVE_ID = "1h2fVtLQJpgLl4_C3MLA_VDuqlJTcAqf6"
# ============================================

if MODEL_DRIVE_ID:
    print("Downloading model from Google Drive...")
    
    # Install gdown for Drive downloads
    !pip install -q gdown
    
    import gdown
    
    # Download directly into the repo's model/ directory
    model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'
    url = f'https://drive.google.com/uc?id={MODEL_DRIVE_ID}'
    
    try:
        gdown.download(url, model_path, quiet=False)
        
        # Verify download
        if os.path.exists(model_path):
            size_mb = os.path.getsize(model_path) / (1024 * 1024)
            if size_mb > 100:  # Should be ~528MB
                print(f"\nModel downloaded successfully: {size_mb:.1f} MB")
                print(f"   Location: {model_path}")
            else:
                print(f"\nModel file seems too small ({size_mb:.1f} MB)")
                print("   Check if the Drive link allows public access")
        else:
            print("\nModel download failed")
            print("   Make sure the file is shared with 'Anyone with the link'")
    except Exception as e:
        print(f"\nError downloading model: {e}")
        print("   Double-check the file ID and sharing permissions")
else:
    print("No model file ID provided")
    print("   Will run baseline-only experiments (no neural ISD prediction)")

# Show model directory contents
print("\nModel directory:")
!ls -lh /content/Spectral_Ratio_Illumination_Demo/model/


## Step 3: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (full version with GUI support)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess

print("Installing dependencies...\n")

# Upgrade pip
print("Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv and scientific computing libraries
print("\nInstalling OpenCV, NumPy, Matplotlib, scikit-image...")
!pip install --quiet opencv-python numpy matplotlib scikit-image scipy

# Install PyTorch with GPU support if available
print("\nInstalling PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\nInstalling remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\nAll dependencies installed successfully!")

# Verify installations
print("\nChecking installed versions:")
import torch
import cv2
import numpy as np
from skimage import __version__ as skimage_version
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   scikit-image: {skimage_version}")

## Step 4: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 5: Run Experiments

culatioThis cell processes all images in `data/images/` with four algorithms.
- Neural ISD prediction
- SR-constrained Retinex
- Baseline Retinex (for comparison)
- SR-based color correction

In [ ]:
import os
import sys

%cd /content/Spectral_Ratio_Illumination_Demo

# Set PYTHONPATH environment variable so subprocess can find modules
os.environ['PYTHONPATH'] = '/content/Spectral_Ratio_Illumination_Demo'

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction")
    print("   Plus: Quality metrics (SSIM + color constancy)")
    print("   Using optimized parameters: iterations=3, sigma=25, distance=1.0\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --gray-world \
        --white-patch \
        --multiscale-retinex \
        --compute-metrics \
        --iterations 5 \
        --sigma 25 \
        --distance 1.0
else:
    print("Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --compute-metrics \
        --iterations 3 \
        --sigma 25

print("\nProcessing complete! Check results/ directory for images and quality_metrics.json")

## Step 6: View Sample Results

Display a few output images to verify processing worked correctly.

## View Quality Metrics

Display the quantitative comparison between your SR-constrained method and the baseline.

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

metrics_file = 'results/quality_metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    print("Quality Metrics Summary\n")
    print("="*70)
    
    # Collect data per method
    method_data = {
        'sr_retinex': {'color_errors': [], 'ssims': [], 'name': 'SR-Constrained Retinex'},
        'baseline_retinex': {'color_errors': [], 'ssims': [], 'name': 'Baseline Retinex'},
        'sr_color_correction': {'color_errors': [], 'ssims': [], 'name': 'SR Color Correction'},
        'gray_world': {'color_errors': [], 'ssims': [], 'name': 'Gray World'},
        'white_patch': {'color_errors': [], 'ssims': [], 'name': 'White Patch'},
        'multiscale_retinex': {'color_errors': [], 'ssims': [], 'name': 'Multi-Scale Retinex'}
    }
    
    for img_name, img_metrics in metrics.items():
        for method in method_data.keys():
            if method in img_metrics:
                method_data[method]['color_errors'].append(img_metrics[method]['color_constancy_error_deg'])
                method_data[method]['ssims'].append(img_metrics[method]['ssim'])
    
    # Print table
    print(f"{'Method':<30} {'Avg Color Error (°)':<20} {'Avg SSIM':<15}")
    print("-"*70)
    
    for method, data in method_data.items():
        if data['color_errors']:
            avg_color = np.mean(data['color_errors'])
            avg_ssim = np.mean(data['ssims'])
            print(f"{data['name']:<30} {avg_color:<20.2f} {avg_ssim:<15.4f}")
    
    print("="*70)
    
    # Visualize comparison - only show methods that have data
    methods_to_plot = [(name, data) for name, data in method_data.items() if data['color_errors']]
    
    if len(methods_to_plot) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Prepare data for plotting
        method_names = [data['name'] for _, data in methods_to_plot]
        color_means = [np.mean(data['color_errors']) for _, data in methods_to_plot]
        ssim_means = [np.mean(data['ssims']) for _, data in methods_to_plot]
        
        # Color palette
        plot_colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']
        bar_colors = plot_colors[:len(methods_to_plot)]
        
        # Color error comparison
        axes[0].bar(method_names, color_means, color=bar_colors, alpha=0.7, edgecolor='black')
        axes[0].set_ylabel('Color Constancy Error (degrees)', fontsize=12)
        axes[0].set_title('Color Preservation\n(Lower is Better)', fontsize=14, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(axis='y', alpha=0.3)
        
        # SSIM comparison
        axes[1].bar(method_names, ssim_means, color=bar_colors, alpha=0.7, edgecolor='black')
        axes[1].set_ylabel('SSIM Score', fontsize=12)
        axes[1].set_title('Structural Similarity\n(Higher is Better)', fontsize=14, fontweight='bold')
        axes[1].set_ylim([0, 1])
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No metrics file found. Make sure Step 5 completed with --compute-metrics flag.")

## View Sample Results (Step 5 outputs)

This displays results from **Step 5** (the main experiment run) for presentation.

**Shows 4 method comparison:**
- Original input (8-bit reference)
- Baseline Retinex (standard method - may shift colors)
- SR-constrained Retinex (new method - preserves colors better)
- SR color correction (simple illumination adjustment)

**Expected Results:**
- SR-constrained should preserve color better than baseline (less color shifts)
- Baseline may over-smooth or change colors unrealistically
- SR correction should brighten shadows while maintaining color relationships

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

results_dir = 'results'

if not os.path.exists(results_dir):
    print("No results found. Run Step 5 first.")
else:
    # Find all result files
    all_files = sorted(os.listdir(results_dir))
    
    # Find all base image names (extract from 8-bit reference images)
    base_names = set()
    for f in all_files:
        if f.endswith('_image_8bit.png'):
            base_names.add(f.replace('_image_8bit.png', ''))
    
    if not base_names:
        print("No processed images found in results/")
    else:
        # Pick first 3 images for visualization
        base_names_list = sorted(base_names)[:3]
        n_images_to_show = len(base_names_list)
        
        print(f"Showing {n_images_to_show} sample image(s) with 6 methods each\n")
        
        # Define all methods to display
        methods = [
            ('_image_8bit.png', 'Original'),
            ('_sr_shifted_vis.png', 'SR Color Correction'),
            ('_baseline_retinex_vis.png', 'Baseline Retinex'),
            ('_sr_retinex_vis.png', 'SR-Constrained Retinex'),
            ('_gray_world_vis.png', 'Gray World'),
            ('_white_patch_vis.png', 'White Patch')
        ]
        
        # Create figure with subplots: n_images rows x 6 methods columns
        fig, axes = plt.subplots(n_images_to_show, 6, figsize=(24, 6 * n_images_to_show))
        
        # Handle case where only 1 image (axes won't be 2D)
        if n_images_to_show == 1:
            axes = axes.reshape(1, -1)
        
        for img_idx, base_name in enumerate(base_names_list):
            for method_idx, (suffix, label) in enumerate(methods):
                filename = f'{base_name}{suffix}'
                img_path = os.path.join(results_dir, filename)
                
                ax = axes[img_idx, method_idx]
                
                if os.path.exists(img_path):
                    img = Image.open(img_path)
                    ax.imshow(img)
                    # Only show method name in top row
                    if img_idx == 0:
                        ax.set_title(label, fontsize=14, fontweight='bold', pad=10)
                else:
                    ax.text(0.5, 0.5, 'Missing', ha='center', va='center', fontsize=12, color='red')
                    if img_idx == 0:
                        ax.set_title(label, fontsize=14, fontweight='bold', pad=10)
                
                ax.axis('off')
                
                # Add image name on left side
                if method_idx == 0:
                    ax.text(-0.1, 0.5, base_name, transform=ax.transAxes,
                           fontsize=11, va='center', ha='right', rotation=0)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nDisplayed {n_images_to_show} sample image(s)")
        print("\nMethod comparison:")
        print("   - SR Color Correction: Best color accuracy (2.59°)")
        print("   - Gray World/White Patch: Highest structural similarity")
        print("   - Retinex methods: Shadow removal with illumination normalization")
        
        if len(base_names) > 3:
            print(f"\nTotal: {len(base_names)} images processed")
            print(f"   Showing first 3, others: {', '.join(sorted(base_names)[3:6])}")